# Phase 5 Calibration and Stability

Champion: **GBM application-only**. LendingClub's `grade`/`sub_grade`/`int_rate` are that
platform's own risk output, so an originator scoring its own applicants has no counterpart
to them. GBM full is kept as a benchmark that measures what those columns add, not as the
model taken forward.

Discrimination is settled (OOT AUC 0.6972, Gini 0.3945). Nothing so far has checked whether
a predicted PD of 0.12 is followed by 12% defaults - which is what every downstream decision
depends on.


In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
import yaml

from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.evaluation.calibration import (
    Calibrator,
    brier_decomposition,
    central_tendency_shift,
    expected_calibration_error,
    pd_to_score,
    reliability_table,
)
from credit_risk.evaluation.stability import population_stability_index, psi_report
from credit_risk.features.build_dataset import (
    APPLICATION_FEATURES,
    application_features,
    assemble_feature_matrix,
)
from credit_risk.models.gbm import predict_gbm, train_gbm

pl.Config.set_tbl_rows(40)

CONFIG_PATH = Path("../configs/base.yaml")
PARAMS_PATH = Path("../configs/gbm_best_params_application.yaml")
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")

In [ ]:
final = assemble_feature_matrix(
    build_target(load_raw_accepted_loans(DATA_PATH), CONFIG_PATH), CONFIG_PATH
)
splits = {
    name: final.filter(pl.col("split") == name) for name in ("train", "validation", "oot_test")
}

features = application_features(final)
params = yaml.safe_load(PARAMS_PATH.read_text())
model, features = train_gbm(splits["train"], splits["validation"], params=params, features=features)

pred = {name: predict_gbm(model, features, df) for name, df in splits.items()}
y = {name: df["default_flag"].to_numpy() for name, df in splits.items()}

for name in splits:
    print(
        name,
        "mean_pd",
        round(float(pred[name].mean()), 4),
        "actual",
        round(float(y[name].mean()), 4),
    )

## 1. Is the champion calibrated out of the box?

`mean_pd` vs `actual` above is the crude check. Train should match almost exactly - the model
fitted it. The gap on validation and OOT is the real question: the model learned an 8.9% base
rate and is being applied to populations running at 10.7% and 11.4%.

`gap` below is predicted minus observed, so positive means the model is too pessimistic.


In [ ]:
for name in ("validation", "oot_test"):
    print(f"--- {name} ---")
    print(reliability_table(y[name], pred[name], n_bins=10))
    print("ECE", round(expected_calibration_error(y[name], pred[name]), 4))
    print(brier_decomposition(y[name], pred[name]), "\n")

## 2. Calibrate on validation, measure on OOT

The calibrator is fitted on 2015 and applied to 2016. Fitting it on train would re-learn the
fit the model already has and report near-perfect calibration that does not exist.

Isotonic and Platt are both fitted so the choice is evidence-based: isotonic corrects any
monotone distortion but cannot extrapolate past its fitted range; Platt only shifts and
rescales, which is more stable in thin tails.


In [ ]:
results = {}
for method in ("platt", "isotonic"):
    calibrator = Calibrator(method).fit(y["validation"], pred["validation"])
    calibrated = calibrator.transform(pred["oot_test"])
    results[method] = calibrated
    print(
        method,
        "ECE",
        round(expected_calibration_error(y["oot_test"], calibrated), 4),
        "mean_pd",
        round(float(calibrated.mean()), 4),
        "min_pd",
        f"{calibrated.min():.2e}",
        "distinct",
        len(np.unique(calibrated)),
    )

print("uncalibrated ECE", round(expected_calibration_error(y["oot_test"], pred["oot_test"]), 4))
print("actual OOT rate ", round(float(y["oot_test"].mean()), 4))

In [ ]:
# Platt is preferred unless isotonic beats it by more than noise: it preserves ranking
# exactly, extrapolates, and cannot collapse a block of loans onto a single PD.
from credit_risk.evaluation.metrics import auc

calibrated = results["platt"]
for method, values in results.items():
    print(
        method,
        "AUC",
        round(auc(y["oot_test"], values), 4),
        "| ECE",
        round(expected_calibration_error(y["oot_test"], values), 4),
    )
print("uncalibrated AUC", round(auc(y["oot_test"], pred["oot_test"]), 4))

## 3. Calibration per term the horizon bias

H=24 truncates the two terms unequally: it captures roughly 60% of eventual 36-month defaults
but only 42% of 60-month ones. A single calibration applied to both therefore understates
60-month risk by more than it understates 36-month risk.

If `gap` differs materially between terms, calibrate per term rather than pooled.


In [ ]:
oot = splits["oot_test"].with_columns(pl.Series("pd_calibrated", calibrated))

for term in (36, 60):
    part = oot.filter(pl.col("term_months") == term)
    yt = part["default_flag"].to_numpy()
    pt = part["pd_calibrated"].to_numpy()
    print(
        f"term={term}  n={part.height}  mean_pd={pt.mean():.4f}  observed={yt.mean():.4f} "
        f"gap={pt.mean() - yt.mean():+.4f}  ECE={expected_calibration_error(yt, pt):.4f}"
    )

In [ ]:
# Per-term calibrators, fitted on validation within each term.
valid = splits["validation"]
for term in (36, 60):
    v = valid.filter(pl.col("term_months") == term)
    o = oot.filter(pl.col("term_months") == term)
    idx_v = valid["term_months"].to_numpy() == term
    idx_o = oot["term_months"].to_numpy() == term
    cal = Calibrator("platt").fit(v["default_flag"].to_numpy(), pred["validation"][idx_v])
    pt = cal.transform(pred["oot_test"][idx_o])
    yt = o["default_flag"].to_numpy()
    print(
        f"term={term}  mean_pd={pt.mean():.4f}  observed={yt.mean():.4f} "
        f"gap={pt.mean() - yt.mean():+.4f}  ECE={expected_calibration_error(yt, pt):.4f}"
    )

## 4. Central tendency and point scale

Calibrating to 2016 is fitting to one vintage. A production PD is usually anchored to a
long-run average instead, which is a pure intercept shift in log-odds: ranking is preserved,
only the level moves. The long-run rate here is the mean across the three observed vintages -
a placeholder, since three vintages is not a cycle.

`pd_to_score` then converts PD to points. A PD without a point scale is not yet a scorecard:
cutoffs, overrides and production monitoring are all expressed in points.


In [ ]:
long_run_rate = float(np.mean([y[name].mean() for name in splits]))
shift = central_tendency_shift(calibrated, long_run_rate)
logits = np.log(np.clip(calibrated, 1e-9, 1 - 1e-9) / (1 - np.clip(calibrated, 1e-9, 1 - 1e-9)))
anchored = 1 / (1 + np.exp(-(logits + shift)))

print(f"long-run rate {long_run_rate:.4f}  log-odds shift {shift:+.4f}")
print(f"mean PD  {calibrated.mean():.4f} -> {anchored.mean():.4f}")
print("AUC unchanged:", round(auc(y["oot_test"], anchored), 4))

scores = pd_to_score(anchored, pdo=20, base_score=600, base_odds=50.0)
print(
    "score range",
    round(float(scores.min())),
    "-",
    round(float(scores.max())),
    " median",
    round(float(np.median(scores))),
)

In [ ]:
# Bad rate by score band - the table a credit committee actually reads.
band = pl.DataFrame({"score": scores, "default_flag": y["oot_test"]}).with_columns(
    (pl.col("score") / 20).floor().cast(pl.Int32).alias("band")
)
band.group_by("band").agg(
    pl.len().alias("n"),
    pl.col("score").min().round(0).alias("score_from"),
    pl.col("default_flag").mean().round(4).alias("bad_rate"),
).sort("band", descending=True)

## 5. Stability (PSI)

Score PSI answers whether the population the model sees in 2016 still resembles 2013-2014.
Feature PSI localises any shift. Bands: <0.10 stable, 0.10-0.25 watch, >0.25 material.

Read these against the discrimination result, not on their own: OOT AUC held at 0.6972, so a
material PSI here would mean the population moved without the score's ranking breaking - a
reason to re-calibrate, not to retrain.


In [ ]:
score_psi = population_stability_index(
    pl.Series(pred["train"]), pl.Series(pred["oot_test"]), n_bins=10
)
print("score PSI train -> oot_test:", round(score_psi, 4))
print(
    "score PSI train -> validation:",
    round(population_stability_index(pl.Series(pred["train"]), pl.Series(pred["validation"])), 4),
)

In [ ]:
report = psi_report(splits["train"], splits["oot_test"], APPLICATION_FEATURES)
print(report.to_pandas().to_string(index=False))
report.write_csv("../docs/psi_application_features.csv")